<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/paperbananamio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip install paperbanana[openai,google,studio]

### Getting Started with Paperbanana

To use `paperbanana` with Gemini or OpenAI models, make sure you have your API keys set in the Colab secrets manager (the key icon on the left).

1. Add `GOOGLE_API_KEY` or `OPENAI_API_KEY`.
2. Enable the "Notebook access" toggle for these keys.

In [2]:
import paperbanana
from google.colab import userdata
import os

# Attempt to set the API key from Colab secrets
try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print("API Key loaded successfully.")
except Exception as e:
    print("Could not find GOOGLE_API_KEY in secrets. Please add it to use Gemini models.")

# Print version to verify installation
print(f"Paperbanana version: {paperbanana.__version__ if hasattr(paperbanana, '__version__') else 'Installed'}")

API Key loaded successfully.
Paperbanana version: 0.1.0


In [ ]:
import os

# Configuring environment overrides from Resources context
os.environ['OPENAI_VLM_MODEL'] = 'gpt-5.2'
os.environ['OPENAI_IMAGE_MODEL'] = 'gpt-image-1.5'

os.environ['GOOGLE_VLM_MODEL'] = 'gemini-2.0-flash'
os.environ['GOOGLE_IMAGE_MODEL'] = 'gemini-3-pro-image-preview'

print("Environment variables for model overrides have been set.")

In [6]:
from paperbanana import PaperBananaPipeline

# Initialize the pipeline
pipeline = PaperBananaPipeline()

# Using the search method within the pipeline
try:
    # In version 0.1.x, search is typically an async method or part of a toolset
    # We will try to call it directly as it is the most common pattern for this class
    results = pipeline.search("Large Language Models for medical diagnosis", limit=3)

    for paper in results:
        print(f"Title: {getattr(paper, 'title', 'N/A')}")
        print(f"Link: {getattr(paper, 'url', 'N/A')}\n")
except AttributeError:
    print("Pipeline initialized, but 'search' method not found directly. Let's check pipeline methods:")
    print([item for item in dir(pipeline) if not item.startswith('_')])
except Exception as e:
    print(f"An error occurred during search: {e}")

2026-05-20 21:23:26 [info     ] Creating VLM provider          model=gemini-2.0-flash provider=gemini
2026-05-20 21:23:26 [info     ] Creating image gen provider    model=gemini-3-pro-image-preview provider=google_imagen
2026-05-20 21:23:26 [info     ] Pipeline initialized           image_gen=google_imagen run_id=run_20260520_212326_0bda83 vlm=gemini
Pipeline initialized, but 'search' method not found directly. Let's check pipeline methods:
['critic', 'generate', 'planner', 'reference_store', 'retriever', 'run_id', 'settings', 'stylist', 'visualizer']


### Generating Content with Paperbanana
Based on your snippet, we can use the `generate` capability. We will define a sample input text and use the pipeline to process it.

In [8]:
from paperbanana import PaperBananaPipeline, GenerationInput

# Initialize the pipeline
pipeline = PaperBananaPipeline()

# Define the input text for generation
sample_text = """
Our architecture follows a standard encoder-decoder structure.
However, we introduce a sparse routing mechanism in the feed-forward layers
to improve efficiency without sacrificing capacity.
"""

# Setup the generation input with all required fields
gen_input = GenerationInput(
    text=sample_text,
    caption="Overview of our encoder-decoder framework",
    source_context="Research paper methodology section",
    communicative_intent="Explain the structural components of the proposed architecture",
    optimize=True,
    auto=True
)

# Run the generation
try:
    print("Starting generation process...")
    output = pipeline.generate(gen_input)

    print("\nGeneration Output Summary:")
    print(output)

    # Display the generated diagram if a path is provided in the output
    if hasattr(output, 'diagram_path') and output.diagram_path:
        from IPython.display import Image, display
        print(f"\nDisplaying generated diagram from: {output.diagram_path}")
        display(Image(filename=output.diagram_path))
except Exception as e:
    print(f"An error occurred during generation: {e}")
    print("\nHint: Ensure your API keys are correctly set in Colab secrets.")

2026-05-20 21:29:16 [info     ] Creating VLM provider          model=gemini-2.0-flash provider=gemini
2026-05-20 21:29:16 [info     ] Creating image gen provider    model=gemini-3-pro-image-preview provider=google_imagen
2026-05-20 21:29:16 [info     ] Pipeline initialized           image_gen=google_imagen run_id=run_20260520_212916_6eebfc vlm=gemini
Starting generation process...

Generation Output Summary:
<coroutine object PaperBananaPipeline.generate at 0x780ea1389f20>


In [9]:
import paperbanana
# Checking for studio or server-related modules in the package
print("Searching for studio-related modules...")
print([m for m in dir(paperbanana) if 'studio' in m.lower() or 'server' in m.lower()])

# Also checking if there's a specific entry point we should know about
try:
    from paperbanana import studio
    print("Studio module found.")
except ImportError:
    print("Studio module not found directly in the root.")

Searching for studio-related modules...
[]
Studio module not found directly in the root.


### Installing Studio Dependencies
If the studio requires extra dependencies, we can install them using the `[studio]` extra.

In [10]:
!pip install 'paperbanana[studio]'

### Launching Paperbanana Studio

This will start the Paperbanana Studio server in the background. Once it starts, you can access it via the generated link.

In [11]:
import threading
import os
from google.colab import output

# Start the studio in a background thread to avoid blocking the cell
def run_studio():
    os.system('paperbanana studio')

thread = threading.Thread(target=run_studio)
thread.start()

# Provide a link to the local server (usually port 8000)
print("Paperbanana Studio is starting...")
print("If it uses port 8000, you can access it here:")
output.serve_kernel_port_as_window(8000)

Paperbanana Studio is starting...
If it uses port 8000, you can access it here:
Try `serve_kernel_port_as_iframe` instead. 


/tmp/ipykernel_2467/786136577.py:3: RuntimeWarning: coroutine 'PaperBananaPipeline.generate' was never awaited
  from google.colab import output


<IPython.core.display.Javascript object>

### Studio CLI Configuration

You can also launch the studio with specific flags to control the network port and the directory where session outputs are saved.

In [28]:
# Example: Launch studio on a custom port with a specific output directory
# !paperbanana studio --port 8080 --output-dir ./my_outputs

### Batch Processing

We will create a manifest file to define multiple generation tasks and then run them in batch mode.

In [12]:
import yaml
import os

# Create an examples directory and dummy files for the manifest to reference
os.makedirs('examples', exist_ok=True)
with open('method1.txt', 'w') as f: f.write('Encoder-decoder architecture details...')
with open('method2.txt', 'w') as f: f.write('Training pipeline steps...')
# Note: paper.pdf would normally be uploaded by the user

# Define the manifest based on the provided resource structure
manifest = {
    'items': [
        {
            'input': 'method1.txt',
            'caption': 'Overview of our encoder-decoder',
            'id': 'fig1'
        },
        {
            'input': 'method2.txt',
            'caption': 'Training pipeline',
            'id': 'fig2'
        },
        {
            'input': 'paper.pdf',
            'caption': 'System overview',
            'id': 'fig3',
            'pdf_pages': '4-9'
        }
    ]
}

# Write the manifest to a YAML file
with open('examples/batch_manifest.yaml', 'w') as f:
    yaml.dump(manifest, f)

print('Updated manifest file created at examples/batch_manifest.yaml')
print('Note: The batch command will attempt to process these items. Ensure method1.txt, method2.txt, and paper.pdf exist.')

Updated manifest file created at examples/batch_manifest.yaml
Note: The batch command will attempt to process these items. Ensure method1.txt, method2.txt, and paper.pdf exist.


### Creating Composite Diagrams
This manifest uses the `composite` layout to combine multiple panels into a single figure (e.g., a 1x3 grid).

In [14]:
import yaml
import os

# Create the encoder input file referenced in the new manifest
with open('method_encoder.txt', 'w') as f:
    f.write('Detailed view of the transformer encoder blocks and attention heads.')

composite_manifest = {
    'composite': {
        'layout': '1x3',
        'labels': 'auto',
        'spacing': 20,
        'label_position': 'bottom',
        'output': 'composite.png'
    },
    'items': [
        {
            'input': 'method_encoder.txt',
            'caption': 'Encoder architecture',
            'id': 'panel_a'
        }
        # Additional panels can be added here to fill the 1x3 grid
    ]
}

with open('examples/composite_manifest.yaml', 'w') as f:
    yaml.dump(composite_manifest, f)

print('Composite manifest created at examples/composite_manifest.yaml')

Composite manifest created at examples/composite_manifest.yaml


### Parameter Sweeps

Sweeps allow you to run the same input against multiple configuration variations (e.g., trying different optimization flags or styles) to see which produces the best result.

In [18]:
import yaml
import os

# Create a sweep manifest
sweep_manifest = {
    'input': 'method_encoder.txt',
    'caption': 'Encoder Sweep Analysis',
    'parameters': {
        'optimize': [True, False],
        'auto': [True, False]
    }
}

with open('examples/sweep_manifest.yaml', 'w') as f:
    yaml.dump(sweep_manifest, f)

print('Sweep manifest created at examples/sweep_manifest.yaml')

Sweep manifest created at examples/sweep_manifest.yaml


In [19]:
!paperbanana sweep --manifest examples/sweep_manifest.yaml

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'sweep'. Did you mean 'setup'?                               │
╰──────────────────────────────────────────────────────────────────────────────╯


### Generating Sweep Reports

Similar to batch reports, you can generate summaries for your parameter sweeps to evaluate the different configurations.

In [20]:
# Example: Generate a sweep report in HTML format
# Replace 'YOUR_SWEEP_ID' with the actual ID from your sweep logs
!paperbanana sweep-report --sweep-id "YOUR_SWEEP_ID" --format html --output sweep_report.html

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'sweep-report'.                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


In [15]:
!paperbanana batch --manifest examples/composite_manifest.yaml --optimize

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'batch'.                                                     │
╰──────────────────────────────────────────────────────────────────────────────╯


In [13]:
!paperbanana batch --manifest examples/batch_manifest.yaml --optimize

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'batch'.                                                     │
╰──────────────────────────────────────────────────────────────────────────────╯


### Generating Batch Reports

After running a batch, you can generate a report to summarize the outputs. Since batch IDs are generated at runtime, make sure to replace the ID below with the one from your logs.

In [16]:
# Example: Generate a markdown report to stdout for the latest batch
# Replace 'YOUR_BATCH_ID' with the actual ID from the batch command output
!paperbanana batch-report --batch-id "YOUR_BATCH_ID" --format markdown

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'batch-report'.                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


In [17]:
# Example: Save an HTML report to a file
!paperbanana batch-report --batch-id "YOUR_BATCH_ID" --format html --output report.html

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'batch-report'.                                              │
╰──────────────────────────────────────────────────────────────────────────────╯


### Batch Plotting

We will create a specific manifest for plotting and then execute the `plot-batch` command.

In [21]:
import yaml
import os

# Create a plot-batch manifest
plot_batch_manifest = {
    'items': [
        {
            'input': 'method_encoder.txt',
            'caption': 'Encoder Visualization',
            'id': 'plot_1'
        },
        {
            'input': 'method1.txt',
            'caption': 'Architecture Layout',
            'id': 'plot_2'
        }
    ]
}

with open('examples/plot_batch_manifest.yaml', 'w') as f:
    yaml.dump(plot_batch_manifest, f)

print('Plot batch manifest created at examples/plot_batch_manifest.yaml')

Plot batch manifest created at examples/plot_batch_manifest.yaml


In [22]:
!paperbanana plot-batch --manifest examples/plot_batch_manifest.yaml --optimize

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'plot-batch'.                                                │
╰──────────────────────────────────────────────────────────────────────────────╯


### Processing Custom Resources

Using the manifest structure provided in the context, we'll create a new manifest and execute the batch plot command.

In [23]:
import yaml
import os

# Define the resource-based manifest from context
resources_manifest = {
    'items': [
        {
            'data': 'path/to/results.csv',
            'intent': 'Bar chart comparing accuracy across models',
            'id': 'fig_acc'
        },
        {
            'data': 'other.json',
            'intent': 'Scatter plot with trend line',
            'aspect_ratio': '16:9'
        }
    ]
}

# Ensure the directory exists and write the manifest
os.makedirs('examples', exist_ok=True)
with open('examples/resources_manifest.yaml', 'w') as f:
    yaml.dump(resources_manifest, f)

print('Resource manifest created at examples/resources_manifest.yaml')
# Note: Ensure the data paths exist before running the CLI command.

Resource manifest created at examples/resources_manifest.yaml


In [24]:
!paperbanana plot-batch --manifest examples/resources_manifest.yaml --optimize

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'plot-batch'.                                                │
╰──────────────────────────────────────────────────────────────────────────────╯


### Full Workflow Orchestration

The `orchestrate` command automates the end-to-end process, from analyzing a PDF paper to generating both method diagrams and data plots.

In [25]:
# Note: Ensure './results' directory and 'paper.pdf' exist for this to run successfully.
!paperbanana orchestrate \
  --paper paper.pdf \
  --data-dir ./results \
  --max-method-figures 4 \
  --max-plot-figures 3 \
  --optimize

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'orchestrate'.                                               │
╰──────────────────────────────────────────────────────────────────────────────╯


### CLI Image Composition

You can also use the CLI directly to combine existing images into a composite layout. This is useful for manual control over panel arrangements.

In [26]:
# Note: Ensure panel_a.png, panel_b.png, and panel_c.png exist before running this.
!paperbanana composite \
  panel_a.png panel_b.png panel_c.png \
  --layout 1x3 \
  --output figure2.png

Usage: paperbanana [OPTIONS] COMMAND [ARGS]...
Try 'paperbanana --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such command 'composite'.                                                 │
╰──────────────────────────────────────────────────────────────────────────────╯


In [35]:
print("Checking available paperbanana CLI commands:")
!paperbanana --help

Checking available paperbanana CLI commands:
                                                                                
 Usage: paperbanana [OPTIONS] COMMAND [ARGS]...                                 
                                                                                
 Generate publication-quality academic illustrations from text.                 
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ────────────────────────────────────────────────────

### Troubleshooting CLI Commands
If the commands like `batch` or `orchestrate` are missing from the help output above, they may be located under a different entry point (like `paperbanana-orchestrate`) or require a specific installation extra.

Let's also check if they are available as separate scripts in the path:

In [36]:
!ls /usr/local/bin/paperbanana*

/usr/local/bin/paperbanana  /usr/local/bin/paperbanana-mcp


### Evaluating Generated Diagrams

The `evaluate` command allows you to compare a generated output against a ground-truth reference, using the original context and caption to assess quality and accuracy.

In [54]:
from paperbanana.agents.critic import CriticAgent
from paperbanana.providers.vlm.gemini import GeminiVLM
import os
import asyncio
from tenacity import retry, stop_after_attempt, wait_exponential

# Ensure prompt directory exists
os.makedirs('prompts/diagram', exist_ok=True)
with open('prompts/diagram/critic.txt', 'w') as f:
    f.write("""You are an expert academic illustrator critique agent.
Review the following diagram based on the description and context.
Description: {{description}}
Context: {{source_context}}
Caption: {{caption}}
""")

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=10))
async def run_evaluation_with_retry(critic):
    return await critic.run(
        "diagram.png",
        "A technical diagram showing an encoder-decoder structure with routing.",
        "Methodology section of the paper",
        "Figure 1: System Architecture"
    )

async def run_final_evaluation():
    vlm = GeminiVLM(model="gemini-2.0-flash", api_key=os.environ.get('GOOGLE_API_KEY'))
    critic = CriticAgent(vlm_provider=vlm, prompt_dir='prompts')

    print("Attempting evaluation with retry logic...")
    try:
        result = await run_evaluation_with_retry(critic)
        print(f"Critique Result: {result}")
    except Exception as e:
        print(f"Evaluation failed after multiple attempts: {e}")

await run_final_evaluation()

Attempting evaluation with retry logic...
2026-05-20 21:44:33 [info     ] Running critic agent           image_path=diagram.png
2026-05-20 21:46:46 [info     ] Running critic agent           image_path=diagram.png
2026-05-20 21:48:58 [info     ] Running critic agent           image_path=diagram.png


CancelledError: 

### Async Pipeline with Custom Settings

You can explicitly configure the VLM and image providers using the `Settings` object and run the pipeline asynchronously.

In [34]:
import asyncio
from paperbanana import PaperBananaPipeline, GenerationInput, DiagramType
from paperbanana.core.config import Settings

# Configure settings - Ensure the [openai] extra is installed for this to work
settings = Settings(
    vlm_provider="openai",
    vlm_model="gpt-4o",
    image_provider="openai",
    image_model="dall-e-3",
    optimize_inputs=True,
    auto_refine=True,
)

try:
    pipeline = PaperBananaPipeline(settings=settings)
    print("Pipeline initialized with OpenAI providers successfully.")
except Exception as e:
    print(f"Initialization failed: {e}")

2026-05-20 21:31:23 [info     ] Creating VLM provider          model=gpt-4o provider=openai
Initialization failed: Unknown VLM provider: openai. Available: gemini, openrouter


### Resuming and Refining Runs

If you have a previous run that needs adjustment, you can load its state and continue with specific feedback to guide the refinement process.

In [46]:
from paperbanana import PaperBananaPipeline, GenerationInput
import asyncio

pipeline = PaperBananaPipeline()

async def generate_with_refinement():
    gen_input = GenerationInput(
        text="Our framework uses a sparse routing mechanism.",
        caption="System Overview",
        source_context="Methodology section",
        communicative_intent="Explain routing",
        optimize=True,
        auto=True
    )

    try:
        print("Starting pipeline... (Note: requires stable API connection)")
        # generate is an awaitable coroutine
        output = await pipeline.generate(gen_input)
        print(f"Final output: {output}")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        print("Hint: Check if your GOOGLE_API_KEY has sufficient quota for gemini-2.0-flash.")

await generate_with_refinement()

2026-05-20 21:36:38 [info     ] Creating VLM provider          model=gemini-2.0-flash provider=gemini
2026-05-20 21:36:38 [info     ] Creating image gen provider    model=gemini-3-pro-image-preview provider=google_imagen
2026-05-20 21:36:38 [info     ] Pipeline initialized           image_gen=google_imagen run_id=run_20260520_213638_874dfc vlm=gemini
Starting pipeline... (Note: requires stable API connection)
2026-05-20 21:36:38 [info     ] Starting generation            context_length=19 diagram_type=methodology run_id=run_20260520_213638_874dfc
2026-05-20 21:36:38 [info     ] Phase 1: Retrieval
2026-05-20 21:36:38 [warning  ] No reference index found       path=data/reference_sets
2026-05-20 21:36:38 [warning  ] No reference candidates available, returning empty list
2026-05-20 21:36:38 [info     ] Phase 1: Planning
2026-05-20 21:36:38 [info     ] Running planner agent          context_length=19 num_examples=0 num_images=0
Pipeline failed: RetryError[<Future at 0x780e9042cda0 state=f

### Model Context Protocol (MCP) Setup

`paperbanana` can be run as an MCP server, allowing it to be used as a tool within supported LLM interfaces (like Claude Desktop). Use the following configuration in your `mcp_config.json` file:

```json
{
  "mcpServers": {
    "paperbanana": {
      "command": "uvx",
      "args": ["--from", "paperbanana[mcp]", "paperbanana-mcp"],
      "env": { "GOOGLE_API_KEY": "your-google-api-key" }
    }
  }
}
```

### Using External Configuration for Generation

You can move your stylistic and model preferences into a YAML file to keep your CLI commands clean and reproducible.

In [ ]:
import yaml

# Create a sample configuration file
config = {
    'vlm_model': 'gemini-2.0-flash',
    'image_model': 'imagen-3.0-capability',
    'style': 'professional_academic',
    'background_color': '#ffffff'
}

with open('my_config.yaml', 'w') as f:
    yaml.dump(config, f)

# Create the input file if it doesn't exist
with open('method.txt', 'w') as f:
    f.write("Our system uses a dual-encoder setup for cross-modal retrieval.")

print('Config and input files prepared.')

In [ ]:
!paperbanana generate \
  --input method.txt \
  --caption "Overview" \
  --config my_config.yaml

### Global Configuration Management

For more complex deployments, you can define a hierarchical configuration that covers model providers, pipeline behavior, and output standards.

In [ ]:
import yaml

# Define the global configuration from the provided resource context
global_config = {
    'vlm': {
        'provider': 'openai',
        'model': 'gpt-4o'  # Using an existing model identifier
    },
    'image': {
        'provider': 'openai_imagen',
        'model': 'dall-e-3'
    },
    'pipeline': {
        'num_retrieval_examples': 10,
        'refinement_iterations': 3,
        'output_resolution': '2k'
    },
    'reference': {
        'path': 'data/reference_sets'
    },
    'output': {
        'dir': 'outputs',
        'save_iterations': True,
        'save_metadata': True
    }
}

with open('global_config.yaml', 'w') as f:
    yaml.dump(global_config, f)

print('Global configuration file saved as global_config.yaml')